<a href="https://colab.research.google.com/github/Sircheikh999/Examen_DataCollection/blob/main/3_1_scraping_et_nettoyage(source_1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installation de Selenium

In [1]:
!pip install google-colab-selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 112.1 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


# Importer packages

In [2]:
# importer packages
import pandas as pd
from selenium.webdriver.common.by import By
import google_colab_selenium as gs
import time

# Lancement du navigateur

In [3]:
# Lancer le navigateur
driver = gs.Chrome()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Scraping des données : Source 1 — Books to Scrape


In [4]:
url = 'https://books.toscrape.com/catalogue/page-1.html'
# ouvrir la page
driver.get(url)

In [5]:
df_final = pd.DataFrame( )
for i in range(1, 51):
  url = f'https://books.toscrape.com/catalogue/page-{i}.html'
  # ouvrir la page
  driver.get(url)
  # containers
  containers = driver.find_elements(By.CSS_SELECTOR, 'article.product_pod')

  data = []
  for container in containers:
    try:

      dic = {
       'title': container.find_element(By.CSS_SELECTOR,'h3 a').get_attribute('title'),
       'price': container.find_element(By.CSS_SELECTOR,'p.price_color').text,
       'availability': container.find_element(By.CSS_SELECTOR, 'p.instock.availability').text.strip(),
       'star_rating': {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}[container.find_element(By.CSS_SELECTOR, 'p.star-rating').get_attribute('class').split()[-1]],
       'book_url': container.find_element(By.CSS_SELECTOR,'h3 a').get_attribute('href')
          }
      data.append(dic)
    except:
      pass

  df = pd.DataFrame(data)
  df_final =pd.concat([df_final, df], axis = 0).reset_index(drop = True)

In [6]:
df_final.head()

,title,price,availability,star_rating,book_url
0,A Light in the Attic,£51.77,In stock,3,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,£53.74,In stock,1,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,£50.10,In stock,1,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,£47.82,In stock,4,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,£54.23,In stock,5,https://books.toscrape.com/catalogue/sapiens-a...


In [7]:
# Création d'un secon dataframe pour récupérer les variables nécessitant l'ouverture de l'URL spécifique à chaque livre
data_details = []

for url in df_final['book_url']:

    driver.get(url)

    try:
        reviews = driver.find_element(By.XPATH,"//th[contains(text(), 'Number of reviews')]/following-sibling::td").text
    except:
        reviews = None

    try:product_description = driver.find_element(By.CSS_SELECTOR,'#product_description + p').text
    except:
        product_description = None

    try:
        product_type = driver.find_element(By.XPATH,"//th[contains(text(), 'Product Type')]/following-sibling::td").text
    except:
        product_type = None

    try:
        tax = driver.find_element(By.XPATH,"//th[contains(text(), 'Tax')]/following-sibling::td"
        ).text
    except:
        tax = None

    dic_details = {
        'book_url': url,
        'reviews': reviews,
        'product_description': product_description,
        'product_type': product_type,
        'tax': tax
    }

    data_details.append(dic_details)

df_details = pd.DataFrame(data_details)

In [8]:
df_details.head()

,book_url,reviews,product_description,product_type,tax
0,https://books.toscrape.com/catalogue/a-light-i...,0,It's hard to imagine a world without A Light i...,Books,£0.00
1,https://books.toscrape.com/catalogue/tipping-t...,0,"""Erotic and absorbing...Written with starling ...",Books,£0.00
2,https://books.toscrape.com/catalogue/soumissio...,0,"Dans une France assez proche de la nôtre, un h...",Books,£0.00
3,https://books.toscrape.com/catalogue/sharp-obj...,0,"WICKED above her hipbone, GIRL across her hear...",Books,£0.00
4,https://books.toscrape.com/catalogue/sapiens-a...,0,From a renowned historian comes a groundbreaki...,Books,£0.00


In [9]:
# Fusion des deux dataframe
df_final = df_final.merge(df_details,on='book_url',how='left')

In [10]:
df_final.head()

,title,price,availability,star_rating,book_url,reviews,product_description,product_type,tax
0,A Light in the Attic,£51.77,In stock,3,https://books.toscrape.com/catalogue/a-light-i...,0,It's hard to imagine a world without A Light i...,Books,£0.00
1,Tipping the Velvet,£53.74,In stock,1,https://books.toscrape.com/catalogue/tipping-t...,0,"""Erotic and absorbing...Written with starling ...",Books,£0.00
2,Soumission,£50.10,In stock,1,https://books.toscrape.com/catalogue/soumissio...,0,"Dans une France assez proche de la nôtre, un h...",Books,£0.00
3,Sharp Objects,£47.82,In stock,4,https://books.toscrape.com/catalogue/sharp-obj...,0,"WICKED above her hipbone, GIRL across her hear...",Books,£0.00
4,Sapiens: A Brief History of Humankind,£54.23,In stock,5,https://books.toscrape.com/catalogue/sapiens-a...,0,From a renowned historian comes a groundbreaki...,Books,£0.00


# Nettoyage des données : Source 1 — Books to Scrape

In [11]:
# localisons les valeurs manquantes
print(df_final.isna().sum())

title                  0
price                  0
availability           0
star_rating            0
book_url               0
reviews                0
product_description    2
product_type           0
tax                    0
dtype: int64


In [12]:
# Remplaçons les valeurs manquantes par la mention : Unknown
df_final['product_description'] = df_final['product_description'].fillna('Unknown')

In [13]:
# Vérification de doublon
print(df_final.duplicated().sum())

0


In [14]:
# Vérification finale
print(df_final.isna().any().sum())

0


# Résultat final du scraping de Source 1 — Books to Scrape

In [15]:
df_final.head()

,title,price,availability,star_rating,book_url,reviews,product_description,product_type,tax
0,A Light in the Attic,£51.77,In stock,3,https://books.toscrape.com/catalogue/a-light-i...,0,It's hard to imagine a world without A Light i...,Books,£0.00
1,Tipping the Velvet,£53.74,In stock,1,https://books.toscrape.com/catalogue/tipping-t...,0,"""Erotic and absorbing...Written with starling ...",Books,£0.00
2,Soumission,£50.10,In stock,1,https://books.toscrape.com/catalogue/soumissio...,0,"Dans une France assez proche de la nôtre, un h...",Books,£0.00
3,Sharp Objects,£47.82,In stock,4,https://books.toscrape.com/catalogue/sharp-obj...,0,"WICKED above her hipbone, GIRL across her hear...",Books,£0.00
4,Sapiens: A Brief History of Humankind,£54.23,In stock,5,https://books.toscrape.com/catalogue/sapiens-a...,0,From a renowned historian comes a groundbreaki...,Books,£0.00
